In [37]:
from apeGmsh import apeGmsh
from apeGmsh import Numberer, FEMData
import numpy as np
import openseespy.opensees as ops
from pathlib import Path

In [38]:
model_iges_path = Path(r'C:\Users\nmora\Github\apeGmsh\acad') / 'tunel_circle.iges'
assert model_iges_path.exists(), model_iges_path

In [39]:


m1 = apeGmsh(model_name="tunnel", verbose=True)
m1.begin()

# 1. Import
m1.model.io.load_step(model_iges_path, highest_dim_only=False)

# 2. Heal the imported geometry — merges duplicate faces/edges
#    within a tolerance so touching volumes become topologically
#    adjacent instead of coincident-but-separate.
m1.model.queries.remove_duplicates(tolerance=1e-3)   # start strict
m1.model.sync()

# 3. Fragment all volumes against each other — this is what makes
#    the interfaces shared OCC faces rather than duplicate pairs.
vol_sel = m1.model.selection.select_volumes()
vol_dts = list(vol_sel.dimtags)
if len(vol_dts) >= 2:
    m1.model.boolean.fragment(
        objects=vol_dts[:1],
        tools=vol_dts[1:],
        dim=3,
    )
m1.model.sync()

# 4. Now mesh — Gmsh will produce conformal nodes at every shared face
m1.mesh.sizing.set_global_size(max_size=500)
m1.mesh.generation.generate(dim=3)

m1.mesh.viewer()

m1.end()

Gmsh version: 4.15.1
[Model] loaded STEP ← tunel_circle.iges  {3: 3, 2: 10, 1: 34, 0: 68}
[Model] remove_duplicates(tolerance=0.001): merged {0: -1, 1: -3, 2: -1} entities (before={0: 12, 1: 17, 2: 10, 3: 3}, after={0: 13, 1: 20, 2: 11, 3: 3})
[Model] OCC kernel synchronised
[Selection] select dim=3 → 3 / 3 entities
[Model] fragment(obj=[(3, 1)], tool=[(3, 2), (3, 3)]) → tags [1, 2, 3]
[Model] OCC kernel synchronised
[Mesh] set_global_size(max=500, min=0.0)
[Mesh] generate(dim=3)

[mesh_scene] Built in 0.27s  (3 actors, 7356 nodes)
  Entities: {1: 20, 2: 11, 3: 3}
  Node setup    : 0.002s
  Actor creation: 0.153s
  Remainder     : 0.116s


In [ ]:
m1 = apeGmsh(model_name="tunnel", verbose=True)
m1.begin()

m1.model.io.load_step(model_iges_path, highest_dim_only=False)

# Heal CAD roundoff + make conformal in two calls
m1.model.queries.remove_duplicates(tolerance=1.0)
m1.model.queries.make_conformal(tolerance=1.0)

# Optional: assign a physical group per volume so you can address them later
for i, (dim, tag) in enumerate(m1.model.selection.select_volumes().dimtags, 1):
    m1.physical.add_volume([tag], name=f"tunnel_part_{i}")

m1.mesh.sizing.set_global_size(1000)
m1.mesh.generation.generate(dim=3)

# Sanity check: should be 0 (conformal already)
fem_before = m1.mesh.queries.get_fem_data(dim=3)
m1.mesh.editing.remove_duplicate_nodes()
fem_after = m1.mesh.queries.get_fem_data(dim=3)
print(f"Duplicate nodes after make_conformal: {fem_before.info.n_nodes - fem_after.info.n_nodes}")

m1.mesh.viewer()

m1.end()


Gmsh version: 4.15.1
[Model] loaded STEP ← tunel_circle.iges  {3: 3, 2: 10, 1: 34, 0: 68}
[Model] remove_duplicates(tolerance=1.0): merged {0: -1, 1: -3, 2: -1} entities (before={0: 12, 1: 17, 2: 10, 3: 3}, after={0: 13, 1: 20, 2: 11, 3: 3})
[Model] make_conformal(dims=[0, 1, 2, 3], tolerance=1.0): entity delta={} (before={0: 13, 1: 20, 2: 11, 3: 3}, after={0: 13, 1: 20, 2: 11, 3: 3})
[Selection] select dim=3 → 3 / 3 entities
[PhysicalGroups] add(dim=3, entities=[1]) → pg_tag=1, name='tunnel_part_1'
[PhysicalGroups] add(dim=3, entities=[2]) → pg_tag=2, name='tunnel_part_2'
[PhysicalGroups] add(dim=3, entities=[3]) → pg_tag=3, name='tunnel_part_3'
[Mesh] set_global_size(max=1000, min=0.0)
[Mesh] generate(dim=3)
[Mesh] get_fem_data(dim=3) → 1543 nodes, 4265 elements, bw=1532
remove_duplicate_nodes: no duplicates found (1543 nodes unchanged)
[Mesh] remove_duplicate_nodes() removed=0
[Mesh] get_fem_data(dim=3) → 1543 nodes, 4265 elements, bw=1532
Duplicate nodes after make_conformal: 0

[m

In [ ]:
m1 = apeGmsh(model_name="tunnel", verbose=True)
m1.begin()

m1.model.io.load_step(model_iges_path, highest_dim_only=False)
m1.model.queries.remove_duplicates(tolerance=1.0)

# Fuse all volumes into a single body
vols = list(m1.model.selection.select_volumes().dimtags)
if len(vols) >= 2:
    m1.model.boolean.fuse(objects=vols[:1], tools=vols[1:])
    m1.model.sync()

print(f"Volumes after fuse: {len(m1.model.selection.select_volumes())}")  # should be 1

m1.physical.add_volume([m1.model.selection.select_volumes().tags[0]], name="tunnel")

m1.mesh.sizing.set_global_size(1000)
m1.mesh.generation.generate(dim=3)
m1.mesh.viewer()

m1.end()

Gmsh version: 4.15.1
[Model] loaded STEP ← tunel_circle.iges  {3: 3, 2: 10, 1: 34, 0: 68}
[Model] remove_duplicates(tolerance=1.0): merged {0: -1, 1: -3, 2: -1} entities (before={0: 12, 1: 17, 2: 10, 3: 3}, after={0: 13, 1: 20, 2: 11, 3: 3})
[Selection] select dim=3 → 3 / 3 entities
[Model] fuse(obj=[(3, 1)], tool=[(3, 2), (3, 3)]) → tags [1]
[Model] OCC kernel synchronised
[Selection] select dim=3 → 1 / 1 entities
Volumes after fuse: 1
[Selection] select dim=3 → 1 / 1 entities
[PhysicalGroups] add(dim=3, entities=[1]) → pg_tag=1, name='tunnel'
[Mesh] set_global_size(max=1000, min=0.0)
[Mesh] generate(dim=3)

[mesh_scene] Built in 0.14s  (3 actors, 1532 nodes)
  Entities: {1: 17, 2: 8, 3: 1}
  Node setup    : 0.001s
  Actor creation: 0.106s
  Remainder     : 0.036s


NameError: name 'q' is not defined